In [53]:
"""Table: Logs

+-------------+---------+
| Column Name | Type    |
+-------------+---------+
| id          | int     |
| num         | varchar |
+-------------+---------+
In SQL, id is the primary key for this table.
id is an autoincrement column starting from 1.
 

Find all numbers that appear at least three times consecutively.

Return the result table in any order.

The result format is in the following example.

 

Example 1:

Input: 
Logs table:
+----+-----+
| id | num |
+----+-----+
| 1  | 1   |
| 2  | 1   |
| 3  | 1   |
| 4  | 2   |
| 5  | 1   |
| 6  | 2   |
| 7  | 2   |
+----+-----+
Output: 
+-----------------+
| ConsecutiveNums |
+-----------------+
| 1               |
+-----------------+
Explanation: 1 is the only number that appears consecutively for at least three times."""

'Table: Logs\n\n+-------------+---------+\n| Column Name | Type    |\n+-------------+---------+\n| id          | int     |\n| num         | varchar |\n+-------------+---------+\nIn SQL, id is the primary key for this table.\nid is an autoincrement column starting from 1.\n \n\nFind all numbers that appear at least three times consecutively.\n\nReturn the result table in any order.\n\nThe result format is in the following example.\n\n \n\nExample 1:\n\nInput: \nLogs table:\n+----+-----+\n| id | num |\n+----+-----+\n| 1  | 1   |\n| 2  | 1   |\n| 3  | 1   |\n| 4  | 2   |\n| 5  | 1   |\n| 6  | 2   |\n| 7  | 2   |\n+----+-----+\nOutput: \n+-----------------+\n| ConsecutiveNums |\n+-----------------+\n| 1               |\n+-----------------+\nExplanation: 1 is the only number that appears consecutively for at least three times.'

In [66]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

spark = SparkSession.builder.getOrCreate()

logs_schema = StructType([
    StructField("id", IntegerType(), False),
    StructField("num", StringType(), True),
])

logs_data = [
    (1, "1"),
    (2, "1"),
    (3, "1"),
    (4, "1"),
    (5, "2"),
    (6, "1"),
    (7, "2"),
    (8, "2"),
]

logs_df = spark.createDataFrame(logs_data, schema=logs_schema)


In [67]:
logs_df.show()

+---+---+
| id|num|
+---+---+
|  1|  1|
|  2|  1|
|  3|  1|
|  4|  1|
|  5|  2|
|  6|  1|
|  7|  2|
|  8|  2|
+---+---+



In [68]:
from pyspark.sql.window import Window
from pyspark.sql.functions import *

In [69]:
window_spec=Window.orderBy("id")

In [ ]:
# has problem like if 4 time same number is comming changed_df added one 
logs_df.withColumn("sec_day",lead('num',1).over(window_spec)).withColumn("third_day",lead('num',2).over(window_spec)).filter((col("num")==col("sec_day"))&(col("num")==col("third_day")))\
    .select("num").show()

+---+
|num|
+---+
|  1|
|  1|
+---+



26/01/06 16:21:55 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/01/06 16:21:55 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/01/06 16:21:55 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/01/06 16:21:55 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/01/06 16:21:55 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


In [72]:
#approach i will say if previous and current is same it will be 1 else 0 and then sum all to get 3 or above
# need to explore solution
from pyspark.sql.functions import lag, sum as spark_sum, when

w = Window.orderBy("id")

df = logs_df.withColumn(
    "grp",
    spark_sum(
        when(col("num") != lag("num").over(w), 1).otherwise(0)
    ).over(w)
)

df.groupBy("num", "grp") \
  .count() \
  .filter(col("count") >= 3) \
  .select("num") \
  .distinct() \
  .show()


26/01/06 16:30:16 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/01/06 16:30:16 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/01/06 16:30:16 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/01/06 16:30:16 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/01/06 16:30:16 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/01/06 16:30:16 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/01/06 1

+---+
|num|
+---+
|  1|
+---+

